# Semana 8 · Consumo de una API pública con `requests`

**Caso:** Spotify — capacidad de streaming y curaduría regional · **Autor:** Julián Andrés Solano Ledesma

Este cuaderno implementa, en pequeño, el paso **Extraer** de la ruta batch del pipeline (ver `diagrama-pipeline-etl.svg`):

1. **Extraer:** consulta la API pública de **MusicBrainz** (JSON por HTTP, sin clave) para los 3 primeros artistas del catálogo normalizado en la Semana 6.
2. **Transformar:** toma solo los campos útiles del JSON y elige el género con más votos de la comunidad.
3. **Cargar:** guarda los 3 registros en `datos/catalogo_musicbrainz.csv`.

El objetivo de negocio es completar la columna `genero_principal` de la tabla `artista`, que quedó vacía porque el CSV de Kaggle no asigna un género único por artista.

## 1. Configuración

In [1]:
import time
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://musicbrainz.org/ws/2"
# MusicBrainz pide identificar la aplicación con un User-Agent descriptivo y un contacto.
HEADERS = {
    "User-Agent": "CORHUILA-CienciaDatos-Semana08/1.0 "
                  "( https://github.com/Drewroots/electiva-vi-ciencia-datos-2026-b-g1 )",
    "Accept": "application/json",
}
PAUSA_SEG = 1.1        # pausa entre peticiones para no saturar el servicio
REINTENTOS = 4         # un buen pipeline reintenta si la fuente falla

RUTA_ARTISTAS = Path("../../06-week/02-optional-activity/datos-normalizados/artista.csv")
RUTA_SALIDA = Path("datos/catalogo_musicbrainz.csv")

## 2. Función de extracción con reintentos

Si el servidor responde 503 (ocupado) o 429 (demasiadas peticiones), se espera y se reintenta con una pausa cada vez mayor. Cualquier otro error se propaga para que el orquestador lo registre y avise.

In [2]:
def get_json(ruta, params=None):
    """GET a la API de MusicBrainz con reintentos; devuelve el JSON como dict."""
    params = {**(params or {}), "fmt": "json"}
    for intento in range(1, REINTENTOS + 1):
        time.sleep(PAUSA_SEG)
        r = requests.get(f"{BASE_URL}/{ruta}", params=params, headers=HEADERS, timeout=15)
        if r.status_code in (429, 503):
            espera = 2 * intento
            print(f"  servidor ocupado ({r.status_code}); reintento {intento} en {espera} s")
            time.sleep(espera)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"La API no respondió después de {REINTENTOS} intentos: {ruta}")

## 3. Extraer: artistas del caso y consulta a la API

In [3]:
artistas = pd.read_csv(RUTA_ARTISTAS).head(3)
artistas

,artist_id,nombre,genero_principal
0,1,Ariana Grande,NaN
1,2,BLACKPINK,NaN
2,3,BTS,NaN


In [4]:
crudos = []
for fila in artistas.itertuples():
    # 3.1 Búsqueda por nombre exacto: devuelve el identificador MBID del artista
    busqueda = get_json("artist/", {"query": f'artist:"{fila.nombre}"', "limit": 1})
    mejor = busqueda["artists"][0]
    # 3.2 Consulta del detalle con géneros (inc=genres)
    detalle = get_json(f"artist/{mejor['id']}", {"inc": "genres"})
    detalle["score"] = mejor["score"]
    crudos.append((fila.artist_id, detalle))
    print(f"OK  {fila.nombre:<15} -> MBID {mejor['id']} (coincidencia {mejor['score']} %)")

OK  Ariana Grande   -> MBID f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387 (coincidencia 100 %)


OK  BLACKPINK       -> MBID 48646387-1664-4c9a-9139-9bfd091b823c (coincidencia 100 %)


OK  BTS             -> MBID 0d79fe8e-ba27-4859-bb8c-2f255f346853 (coincidencia 100 %)


Así llega un registro de la API: JSON **semiestructurado**, con campos anidados (`area`, `life-span`, `genres`). Se muestran solo algunas claves y los 3 géneros con más votos:

In [5]:
import json

_, ejemplo = crudos[0]
vista = {k: ejemplo[k] for k in ("id", "name", "type", "country", "life-span")}
vista["genres (top 3 por votos)"] = [
    {"name": g["name"], "count": g["count"]}
    for g in sorted(ejemplo["genres"], key=lambda g: g["count"], reverse=True)[:3]
]
print(json.dumps(vista, indent=2, ensure_ascii=False))

{
  "id": "f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387",
  "name": "Ariana Grande",
  "type": "Person",
  "country": "US",
  "life-span": {
    "ended": false,
    "begin": "1993-06-26",
    "end": null
  },
  "genres (top 3 por votos)": [
    {
      "name": "pop",
      "count": 21
    },
    {
      "name": "r&b",
      "count": 6
    },
    {
      "name": "dance-pop",
      "count": 4
    }
  ]
}


## 4. Transformar: del JSON a una tabla

Por cada artista se conserva el MBID, el tipo, el país (ISO 3166-1), el año de inicio y el **género con más votos**. Se valida que la búsqueda haya coincidido al 100 % para no asignar el género de otro artista.

In [6]:
def genero_principal(generos):
    """Género con más votos de la comunidad; None si no hay géneros."""
    return max(generos, key=lambda g: g["count"])["name"] if generos else None


registros = [
    {
        "artist_id": artist_id,                       # clave del modelo de la Semana 6
        "mbid": d["id"],                              # identificador en MusicBrainz
        "nombre": d["name"],
        "tipo": d.get("type"),
        "pais_iso": d.get("country"),
        "anio_inicio": (d.get("life-span", {}).get("begin") or "")[:4] or None,
        "genero_principal": genero_principal(d.get("genres", [])),
        "coincidencia_pct": d["score"],
    }
    for artist_id, d in crudos
]
catalogo = pd.DataFrame(registros)

assert (catalogo["coincidencia_pct"] == 100).all(), "Alguna búsqueda no coincidió exactamente"
assert catalogo["artist_id"].is_unique, "artist_id debe ser único (PK)"
catalogo

,artist_id,mbid,nombre,tipo,pais_iso,anio_inicio,genero_principal,coincidencia_pct
0,1,f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387,Ariana Grande,Person,US,1993,pop,100
1,2,48646387-1664-4c9a-9139-9bfd091b823c,BLACKPINK,Group,KR,2016,k-pop,100
2,3,0d79fe8e-ba27-4859-bb8c-2f255f346853,BTS,Group,KR,2013,k-pop,100


## 5. Cargar: los 3 registros quedan listos para el Data Warehouse

In [7]:
RUTA_SALIDA.parent.mkdir(exist_ok=True)
catalogo.drop(columns="coincidencia_pct").to_csv(RUTA_SALIDA, index=False)
print(f"Guardados {len(catalogo)} registros en {RUTA_SALIDA}")
pd.read_csv(RUTA_SALIDA)

Guardados 3 registros en datos\catalogo_musicbrainz.csv


,artist_id,mbid,nombre,tipo,pais_iso,anio_inicio,genero_principal
0,1,f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387,Ariana Grande,Person,US,1993,pop
1,2,48646387-1664-4c9a-9139-9bfd091b823c,BLACKPINK,Group,KR,2016,k-pop
2,3,0d79fe8e-ba27-4859-bb8c-2f255f346853,BTS,Group,KR,2013,k-pop


## 6. Conclusión

- La API entregó los datos en **JSON** por HTTP, el formato habitual de una API.
- En la ruta batch del pipeline, este código se ejecutaría una vez al día como una tarea de **Airflow**: el catálogo cambia lento y no necesita streaming.
- El campo `genero_principal` completa la columna que quedó vacía en la tabla `artista` de la Semana 6, enriqueciendo el modelo para la decisión de **curaduría por región**.